In [1]:
import sys
sys.path.insert(0, '../lib')

import pandas as pd

import common_data

# Table 1 for NPC categories

1. N samples
2. N patients
1. sex
2. age
3. mortality
4. sofa score
5. days on ventilator


In [14]:
CAT_NAMES = {
    'NI': 'Not infected',
    'Infection': 'Extrapulmonary\ninfection',
    'Inflammation': 'Non-infectious\nalveolitis'
}

In [18]:
ehr = pd.read_csv(common_data.CLINICAL, index_col=0)
ehr_labels = pd.read_csv(common_data.CLINICAL_LABELS, index_col=0)

In [19]:
ehr = ehr.merge(ehr_labels, left_index=True, right_index=True, suffixes=('', '_ehr'))

In [20]:
idx = ehr.bal_barcode.isin(common_data.NPC_CATEGORIES.keys())
npc = ehr.loc[idx].copy()

In [21]:
npc.patient.duplicated().sum()

2

In [22]:
npc['NPC_type'] = npc.bal_barcode.replace(common_data.NPC_CATEGORIES).replace(CAT_NAMES)

In [31]:
result = []
for t in npc.NPC_type.unique():
    idx = npc.NPC_type.eq(t)
    idx2 = ~npc.patient[idx].duplicated()
    info = dict(
        type=t,
        n_samples=idx.sum(),
        n_patients=npc.patient[idx].nunique(),
        age=npc.loc[idx].Age[idx2].quantile([0.25, 0.5, 0.75]).values,
        n_female=npc.loc[idx].Gender[idx2].eq('Female').sum(),
        died=npc.loc[idx].Binary_outcome[idx2].sum(),
        sofa=npc.loc[idx].SOFA_score.quantile([0.25, 0.5, 0.75]).values,
        days_on_vent=npc.loc[idx].days_on_ventilator.quantile([0.25, 0.5, 0.75]).values
    )
    result.append(info)
result = pd.DataFrame(result)

In [36]:
result['pct_female'] = result.n_female / result.n_patients * 100
result['pct_died'] = result.died / result.n_patients * 100

In [39]:
def fmt(x):
    return f'{x[1]} [{x[0]}, {x[2]}]'
result.age = result.age.apply(fmt)
result.sofa = result.sofa.apply(fmt)
result.days_on_vent = result.days_on_vent.apply(fmt)

In [40]:
result

,type,n_samples,n_patients,age,n_female,died,sofa,days_on_vent,pct_female,pct_died
0,Extrapulmonary\ninfection,6,6,"47.0 [42.5, 53.75]",4,5,"10.5 [8.0, 14.5]","2.0 [2.0, 2.0]",66.666667,83.333333
1,Non-infectious\nalveolitis,14,13,"53.0 [45.0, 67.0]",6,7,"12.0 [8.25, 14.0]","2.0 [1.0, 2.0]",46.153846,53.846154
2,Not infected,6,6,"68.5 [60.25, 72.25]",4,1,"9.5 [8.25, 10.75]","1.5 [1.0, 2.0]",66.666667,16.666667
